# 〰️ Atmosphäre-Analyse
## Layer 6 – Resonanzfeld / Schumann-Resonanz

| Layer | Name | Status |
|-------|------|--------|
| 0 | Externe kosmische Einflüsse | ✅ |
| 1 | Planetarer Grundkörper | ✅ |
| 2 | Oberfläche & Kontaktzone | ✅ |
| 3 | Atmosphäre / Wetter / Konvektion | ✅ |
| 4 | Ionosphäre | ✅ |
| 5 | Global Electric Circuit | ✅ |
| **6** | **Resonanzfeld / Schumann-Resonanz** | **← dieser Layer** |
| 7 | Earth Field State Engine | ⬜ |
| 8 | Research / Hypothesen | ⬜ |

> **Kernidee:** Schumann-Resonanz ist nicht nur eine Frequenz, sondern ein Ausdruck:
> - globaler Blitzaktivität (Layer 3)
> - der Earth-Ionosphere Cavity (Layer 4)
> - veränderter Ausbreitungsbedingungen (Layer 4 + 5)
> - gekoppelter Systemzustände (alle Layer)
>
> **Zentrale Methode (aus Layer 4):** Vergleich `geometric_delta` ↔ `expected_real_delta` → Diagnosemechanismus für nicht-geometrische Faktoren.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import datetime, json, math
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# --- Projektpfade (CWD-unabhaengig, ohne pip install) ---
import sys, pathlib
_root = pathlib.Path.cwd().resolve()
while not (_root / '.project-root').exists() and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root / 'src'))
from atmosphere.paths import layer_state

print(f'Analysedatum: {datetime.datetime.utcnow().isoformat()}Z')

# Alle vorherigen Layer laden
context = {}
for n in [0, 1, 2, 3, 4, 5]:
    try:
        with open(layer_state(n), encoding='utf-8') as f:
            context[n] = json.load(f)
        print(f'  Layer {n}: {context[n]["level"].upper():8}  Score={context[n]["score"]}')
    except FileNotFoundError:
        print(f'  Layer {n}: nicht gefunden')
        context[n] = None

L0, L1, L2, L3, L4, L5 = (context.get(n) for n in range(6))

# Schlüsselwerte aus Upstream-Layern
# L3: Generator (Blitze, CAPE)
l3_thunder_score = L3.get('components',{}).get('Gewitteraktivität',{}).get('score') if L3 else None
l3_cape          = L3.get('raw_values',{}).get('CAPE_mean_Jkg') if L3 else None
l3_n_thunder     = L3.get('raw_values',{}).get('thunder_points_WMO', 0) if L3 else 0
l3_schumann_pot  = L3.get('raw_values',{}).get('schumann_potential') if L3 else None
l3_messpunkte    = L3.get('raw_values',{}).get('messpunkte', []) if L3 else []

# L4: Cavity-Geometrie
l4_cavity_h      = L4.get('resonance_system',{}).get('cavity_height_km', 80.0) if L4 else 80.0
l4_cavity_delta  = L4.get('resonance_system',{}).get('cavity_delta_km', 0.0) if L4 else 0.0
l4_geom_freq     = L4.get('resonance_system',{}).get('schumann_frequencies_Hz',{}) if L4 else {}
l4_geom_delta_mhz= L4.get('resonance_system',{}).get('schumann_delta_mHz',{}) if L4 else {}
l4_day_night     = L4.get('resonance_system',{}).get('day_night', 'unknown') if L4 else 'unknown'
l4_kp            = L4.get('raw_values',{}).get('Kp_current') if L4 else None
l4_xray_class    = L4.get('raw_values',{}).get('xray_class') if L4 else None
l4_ioniz_score   = L4.get('components',{}).get('Ionisierungsgrad (F10.7)',{}).get('score') if L4 else None

# L5: GEC-Zustand
l5_v_iono        = L5.get('gec_state',{}).get('V_ionosphere_kV', 300.0) if L5 else 300.0
l5_delta_v_pct   = L5.get('gec_state',{}).get('delta_V_pct', 0.0) if L5 else 0.0
l5_generator     = L5.get('gec_state',{}).get('generator_strength', 1.0) if L5 else 1.0
l5_score         = L5.get('score') if L5 else None

# UT-Stunde für Tagesmuster (Schumann-Chimneys)
utc_hour = datetime.datetime.utcnow().hour

print(f'\n  L3 Generator:  thunder={l3_thunder_score}  CAPE={l3_cape}  schumann_pot={l3_schumann_pot}')
print(f'  L4 Cavity:     h={l4_cavity_h} km (Δ {l4_cavity_delta:+.1f})  {l4_day_night}  Kp={l4_kp}')
print(f'  L5 GEC:        V_iono={l5_v_iono} kV (Δ {l5_delta_v_pct:+.1f}%)  generator={l5_generator}')
print(f'  UTC-Stunde:    {utc_hour}h')

---
## 1. Empirische Referenz vs. geometrisches Modell

Schumann-Resonanz hat zwei verschiedene Bezugsrahmen — die Trennung ist zentral:

| Modus | Empirisch beobachtet | Geometrisches Modell (h=80 km) | Differenz |
|-------|---------------------|-------------------------------|-----------|
| SR-1  | **7.83 Hz** (Q≈4–5) | 10.46 Hz | −2.63 Hz |
| SR-2  | **14.3 Hz** | 18.12 Hz | −3.82 Hz |
| SR-3  | **20.8 Hz** | 25.62 Hz | −4.82 Hz |
| SR-4  | **27.3 Hz** | 33.08 Hz | −5.78 Hz |
| SR-5  | **33.8 Hz** | 40.51 Hz | −6.71 Hz |

**Warum die Differenz?** Das idealisierte Cavity-Modell vernachlässigt:
- endliche Leitfähigkeit der Ionosphäre (D-Schicht)
- Dämpfung durch ionosphärische Verluste
- effektive Ausbreitungsgeschwindigkeit < c
- Tag/Nacht-Asymmetrie

**→ Konsequenz:** Layer 6 verwendet die **empirischen Referenzwerte** als physikalische Basis, nicht die geometrischen aus Layer 4.

**→ Diagnosemechanismus:** Das geometrische Δ aus Layer 4 (~+16 mHz pro 10 km) ist die Obergrenze des reinen Cavity-Effekts. Reale Frequenzverschiebungen müssen mit diesem Δ verglichen werden, um nicht-geometrische Treiber zu identifizieren.

In [ ]:
# ============================================================
# EMPIRISCHE SCHUMANN-REFERENZWERTE
# Quelle: Nickolaenko & Hayakawa (2014), gemittelt langjährig
# ============================================================

SR_REF = {
    1: {'freq_Hz': 7.83,  'amplitude_pT': 1.00, 'Q_factor': 4.5},
    2: {'freq_Hz': 14.3,  'amplitude_pT': 0.45, 'Q_factor': 4.8},
    3: {'freq_Hz': 20.8,  'amplitude_pT': 0.30, 'Q_factor': 5.0},
    4: {'freq_Hz': 27.3,  'amplitude_pT': 0.20, 'Q_factor': 5.2},
    5: {'freq_Hz': 33.8,  'amplitude_pT': 0.13, 'Q_factor': 5.4},
}

# Drei Schumann-Chimneys (Hauptgewitterzentren der Erde)
CHIMNEYS = {
    'Asien/Maritime':  {'lon_center': 100, 'peak_UT': 8,  'color': '#F2A623'},
    'Afrika':          {'lon_center': 25,  'peak_UT': 14, 'color': '#E85D24'},
    'Amerika':         {'lon_center': -75, 'peak_UT': 20, 'color': '#534AB7'},
}

print('SCHUMANN-REFERENZWERTE (empirisch, langjähriges Mittel)')
print('=' * 62)
print(f'  {"Modus":<6} {"Freq [Hz]":<11} {"Amp [pT]":<11} {"Q-Faktor":<10}')
print('-' * 62)
for n, ref in SR_REF.items():
    print(f'  SR-{n}   {ref["freq_Hz"]:<11.2f} {ref["amplitude_pT"]:<11.2f} {ref["Q_factor"]:<10.1f}')
print('=' * 62)
print('\nTages-Chimneys (Hauptgewitterzentren):')
for name, c in CHIMNEYS.items():
    active = abs(((utc_hour - c['peak_UT']) + 12) % 24 - 12) < 4
    print(f'  {name:<18} Peak UTC {c["peak_UT"]:>2}h   {"AKTIV" if active else "..."}')

---
## 2. Modulationsmodell: erwartete reale Schumann-Werte

In [ ]:
# ============================================================
# MODULATIONSMODELL
# Erwartete Schumann-Werte = Referenz × Modulatoren aus L3/L4/L5
# ============================================================

# === FREQUENZMODULATION ===
# Aus Layer 4: geometrisches Delta (sehr klein, ~16-51 mHz pro 10 km)
# Plus realer Cavity-Effekt (Tag/Nacht, ionosphärische Verluste)
geom_delta_mhz = {n: l4_geom_delta_mhz.get(f'SR_{n}', 0.0) for n in [1,2,3,4]}

# Tag/Nacht-Effekt (empirisch beobachtet, ~50-100 mHz Verschiebung)
# Tag: niedrigere SR (kleinere effektive Cavity-Höhe), Nacht: höhere SR
day_night_shift_mhz = -50 if l4_day_night == 'day' else +50 if l4_day_night == 'night' else 0

# Ionosphärische Störung (Kp, X-Ray) → Frequenzverschiebung
kp_val = l4_kp if l4_kp is not None else 2.0
kp_shift_mhz = -kp_val * 5  # je höher Kp, desto niedrigere Frequenzen
xray_shift_mhz = -30 if l4_xray_class in ['M', 'X'] else 0

# === AMPLITUDENMODULATION ===
# Aus Layer 3: Gewitteraktivität (Generator)
# Aus Layer 5: GEC-Potential (~Generator-Stärke)
amp_mod = (l5_generator or 1.0) * (1.0 + (l3_thunder_score or 0.15) * 0.5)

# Tagesvariation: ~30% Variation je nach Chimney
# Maximum wenn ein Chimney im Peak ist
chimney_activity = 0.0
active_chimneys = []
for name, c in CHIMNEYS.items():
    distance = abs(((utc_hour - c['peak_UT']) + 12) % 24 - 12)
    activity = max(0, 1 - distance / 6)  # voller Beitrag wenn distance=0, null bei distance>=6
    chimney_activity += activity
    if activity > 0.3:
        active_chimneys.append((name, activity))
chimney_factor = 0.85 + chimney_activity * 0.10  # 0.85–1.15
amp_mod_total = amp_mod * chimney_factor

# === Q-FAKTOR-MODULATION ===
# Q hängt ab von ionosphärischer Leitfähigkeit
# Hohes F10.7 → niedrigere D-Schicht → höhere Verluste → niedrigeres Q
q_mod = 1.0 - (l4_ioniz_score or 0.3) * 0.15  # bis -15% bei sehr aktiver Sonne
if l4_xray_class in ['M', 'X']:
    q_mod *= 0.7  # massive Q-Verschlechterung bei großen Flares

# === ERWARTETE SCHUMANN-WERTE BERECHNEN ===
expected = {}
for n, ref in SR_REF.items():
    # Frequenzverschiebung (mHz → Hz)
    total_shift_mhz = (geom_delta_mhz.get(n, 0) +
                       day_night_shift_mhz +
                       kp_shift_mhz +
                       xray_shift_mhz)
    expected[n] = {
        'freq_Hz':       round(ref['freq_Hz'] + total_shift_mhz / 1000, 4),
        'freq_shift_mHz':round(total_shift_mhz, 1),
        'amplitude_pT':  round(ref['amplitude_pT'] * amp_mod_total, 4),
        'amplitude_factor': round(amp_mod_total, 3),
        'Q_factor':      round(ref['Q_factor'] * q_mod, 3),
        'Q_factor_change': round((q_mod - 1) * 100, 1),
    }

print('ERWARTETE SCHUMANN-WERTE (Referenz × Modulatoren)')
print('=' * 78)
print(f'  {"Modus":<6} {"Freq [Hz]":<11} {"Δf [mHz]":<11} {"Amp [pT]":<11} {"Q":<8} {"ΔQ [%]":<8}')
print('-' * 78)
for n, e in expected.items():
    print(f'  SR-{n}   {e["freq_Hz"]:<11.4f} {e["freq_shift_mHz"]:<+11.1f} '
          f'{e["amplitude_pT"]:<11.4f} {e["Q_factor"]:<8.2f} {e["Q_factor_change"]:<+8.1f}')
print('=' * 78)

print('\nMODULATOR-AUFSCHLÜSSELUNG')
print(f'  Frequenz-Shifts (additiv):')
print(f'    geometrisch (Layer 4):      SR-1 {geom_delta_mhz.get(1,0):+.1f} mHz ... SR-4 {geom_delta_mhz.get(4,0):+.1f} mHz')
print(f'    Tag/Nacht ({l4_day_night}):  {day_night_shift_mhz:+.0f} mHz')
print(f'    Kp-Effekt (Kp={kp_val:.1f}):  {kp_shift_mhz:+.1f} mHz')
print(f'    X-Ray ({l4_xray_class}-Klasse): {xray_shift_mhz:+.0f} mHz')
print(f'  Amplituden-Faktor: {amp_mod_total:.3f}')
print(f'    Generator (L5): {l5_generator:.3f}')
print(f'    Gewitter (L3):  {(1.0 + (l3_thunder_score or 0.15) * 0.5):.3f}')
print(f'    Chimney-Faktor: {chimney_factor:.3f}')
print(f'  Q-Faktor: {q_mod:.3f} (Ionisierung + X-Ray)')
print(f'\nAktive Chimneys: {[c[0] for c in active_chimneys] if active_chimneys else "–"}')

---
## 3. Visualisierungen

In [ ]:
# ============================================================
# SCHUMANN-SPEKTRUM (modelliert)
# Lorentz-Kurven für jeden Modus, mit Modulation
# ============================================================

freqs = np.linspace(3, 40, 800)

def lorentzian(f, f0, A, Q):
    """Normierte Lorentz-Kurve mit Q-Faktor und Amplitude A"""
    gamma = f0 / (2 * Q)
    return A * gamma**2 / ((f - f0)**2 + gamma**2)

spec_ref = np.zeros_like(freqs)
spec_exp = np.zeros_like(freqs)
for n, ref in SR_REF.items():
    spec_ref += lorentzian(freqs, ref['freq_Hz'], ref['amplitude_pT'], ref['Q_factor'])
    e = expected[n]
    spec_exp += lorentzian(freqs, e['freq_Hz'], e['amplitude_pT'], e['Q_factor'])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=freqs, y=spec_ref, mode='lines', name='Referenz (empirisch)',
    line=dict(color='#888780', width=1.5, dash='dot')
))
fig.add_trace(go.Scatter(
    x=freqs, y=spec_exp, mode='lines', name='Erwartet (moduliert)',
    line=dict(color='#F2A623', width=2.2),
    fill='tozeroy', fillcolor='rgba(242,166,35,0.13)'
))

# Modi-Markierungen
for n, ref in SR_REF.items():
    fig.add_annotation(
        x=ref['freq_Hz'], y=ref['amplitude_pT'] * 1.1,
        text=f'SR-{n}<br>{ref["freq_Hz"]:.2f} Hz',
        showarrow=False, font=dict(size=9, color='#888780'),
    )

fig.update_layout(
    title=dict(text='Schumann-Spektrum: Empirisch vs. moduliert erwartet', font=dict(size=14)),
    xaxis=dict(title='Frequenz [Hz]', range=[3, 40], gridcolor='#222244'),
    yaxis=dict(title='Amplitude [pT]', gridcolor='#222244'),
    height=380, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

In [ ]:
# ============================================================
# DREI-SPALTEN-VERGLEICH PRO MODUS:
# Frequenz | Amplitude | Q-Faktor (jeweils Ref vs. Erwartet)
# ============================================================

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=['Frequenz [Hz]', 'Amplitude [pT]', 'Q-Faktor'],
    horizontal_spacing=0.10
)

modes = list(SR_REF.keys())
labels = [f'SR-{n}' for n in modes]

# Frequenz
fig.add_trace(go.Bar(
    name='Referenz', x=labels, y=[SR_REF[n]['freq_Hz'] for n in modes],
    marker_color='#5F5E5A', opacity=0.75, showlegend=True
), row=1, col=1)
fig.add_trace(go.Bar(
    name='Erwartet', x=labels, y=[expected[n]['freq_Hz'] for n in modes],
    marker_color='#F2A623', opacity=0.90,
    text=[f'{expected[n]["freq_Hz"]:.2f}' for n in modes],
    textposition='outside', textfont=dict(color='white', size=9),
    showlegend=True
), row=1, col=1)

# Amplitude
fig.add_trace(go.Bar(
    x=labels, y=[SR_REF[n]['amplitude_pT'] for n in modes],
    marker_color='#5F5E5A', opacity=0.75, showlegend=False
), row=1, col=2)
fig.add_trace(go.Bar(
    x=labels, y=[expected[n]['amplitude_pT'] for n in modes],
    marker_color='#E85D24', opacity=0.90,
    text=[f'{expected[n]["amplitude_pT"]:.2f}' for n in modes],
    textposition='outside', textfont=dict(color='white', size=9),
    showlegend=False
), row=1, col=2)

# Q-Faktor
fig.add_trace(go.Bar(
    x=labels, y=[SR_REF[n]['Q_factor'] for n in modes],
    marker_color='#5F5E5A', opacity=0.75, showlegend=False
), row=1, col=3)
fig.add_trace(go.Bar(
    x=labels, y=[expected[n]['Q_factor'] for n in modes],
    marker_color='#534AB7', opacity=0.90,
    text=[f'{expected[n]["Q_factor"]:.2f}' for n in modes],
    textposition='outside', textfont=dict(color='white', size=9),
    showlegend=False
), row=1, col=3)

fig.update_layout(
    title=dict(text='Schumann-Modi: Empirische Referenz vs. erwarteter modulierter Wert', font=dict(size=13)),
    barmode='group',
    height=400, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=50, r=30, t=70, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

In [ ]:
# ============================================================
# TAGESMUSTER – Schumann-Chimneys (UTC-Diurnalkurve)
# ============================================================

ut_range = list(range(0, 24))

def chimney_curve(peak_ut, ut):
    """Glockenkurve um Peak-UT mit Halbwertsbreite ~6h"""
    distance = abs(((ut - peak_ut) + 12) % 24 - 12)
    return max(0, math.exp(-(distance / 5)**2))

fig = go.Figure()
total_curve = [0.0] * 24
for name, c in CHIMNEYS.items():
    curve = [chimney_curve(c['peak_UT'], h) for h in ut_range]
    total_curve = [t + v for t, v in zip(total_curve, curve)]
    fig.add_trace(go.Scatter(
        x=ut_range, y=curve,
        mode='lines', name=name,
        line=dict(color=c['color'], width=2),
        fill='tozeroy', fillcolor=c['color'].replace('#', 'rgba(') if False else None,
        opacity=0.7,
    ))
fig.add_trace(go.Scatter(
    x=ut_range, y=total_curve,
    mode='lines', name='Globale Aktivität',
    line=dict(color='white', width=2.5, dash='dash')
))

# Aktuelle UTC-Stunde markieren
fig.add_vline(x=utc_hour, line_color='#F2A623', line_width=2,
              annotation_text=f'jetzt: {utc_hour}h UTC',
              annotation_font=dict(color='#F2A623', size=10))

fig.update_layout(
    title=dict(text='Schumann-Chimneys: Tagesmuster (UTC) der globalen Blitzaktivität', font=dict(size=14)),
    xaxis=dict(title='UTC-Stunde', tickmode='linear', tick0=0, dtick=2,
               range=[0, 23], gridcolor='#222244'),
    yaxis=dict(title='Relative Anregungsstärke', gridcolor='#222244'),
    height=350, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

In [ ]:
# ============================================================
# DELTA-DIAGNOSE: geometrisch (L4) vs. nicht-geometrisch (modulatorisch)
# Kernfrage: Welcher Anteil der erwarteten Frequenzverschiebung
# kommt aus reiner Cavity-Geometrie, welcher aus anderen Treibern?
# ============================================================

modes_d = list(SR_REF.keys())
geom_components   = [geom_delta_mhz.get(n, 0)   for n in modes_d]
daynight_components = [day_night_shift_mhz       for _ in modes_d]
kp_components     = [kp_shift_mhz                for _ in modes_d]
xray_components   = [xray_shift_mhz              for _ in modes_d]
totals            = [expected[n]['freq_shift_mHz'] for n in modes_d]

fig = go.Figure()
fig.add_trace(go.Bar(name='Geometrisch (L4)',
    x=[f'SR-{n}' for n in modes_d], y=geom_components,
    marker_color='#888780', opacity=0.85))
fig.add_trace(go.Bar(name=f'Tag/Nacht ({l4_day_night})',
    x=[f'SR-{n}' for n in modes_d], y=daynight_components,
    marker_color='#378ADD', opacity=0.85))
fig.add_trace(go.Bar(name=f'Kp ({kp_val:.1f})',
    x=[f'SR-{n}' for n in modes_d], y=kp_components,
    marker_color='#7F77DD', opacity=0.85))
fig.add_trace(go.Bar(name=f'X-Ray ({l4_xray_class})',
    x=[f'SR-{n}' for n in modes_d], y=xray_components,
    marker_color='#E85D24', opacity=0.85))

# Total als Linie
fig.add_trace(go.Scatter(
    x=[f'SR-{n}' for n in modes_d], y=totals,
    mode='lines+markers+text', name='Summe',
    line=dict(color='#F2A623', width=2.5),
    marker=dict(size=10, color='#F2A623'),
    text=[f'{t:+.0f}' for t in totals],
    textposition='top center',
    textfont=dict(color='#F2A623', size=10)
))

fig.add_hline(y=0, line_color='#888780', line_width=0.8)
fig.update_layout(
    title=dict(text='Delta-Diagnose: Frequenzverschiebung aufgeschlüsselt nach Treiber [mHz]', font=dict(size=14)),
    barmode='relative',
    yaxis=dict(title='Frequenzverschiebung [mHz]', gridcolor='#222244'),
    height=380, plot_bgcolor='#1a1a2e', paper_bgcolor='rgba(0,0,0,0)',
    margin=dict(l=60, r=30, t=55, b=50),
    legend=dict(bgcolor='rgba(20,20,40,0.85)', font=dict(color='white'))
)
fig.show()

# Interpretation
geom_total_sr1     = abs(geom_components[0])
non_geom_total_sr1 = abs(daynight_components[0]) + abs(kp_components[0]) + abs(xray_components[0])
ratio = non_geom_total_sr1 / max(geom_total_sr1, 1.0)
print(f'\nSR-1 Treiber-Verhältnis:')
print(f'  geometrisch:     {geom_total_sr1:.1f} mHz')
print(f'  nicht-geometrisch: {non_geom_total_sr1:.1f} mHz')
print(f'  Verhältnis:      {ratio:.1f}x')
if ratio > 3:
    print('  → Nicht-geometrische Faktoren dominieren deutlich (Verifikation der Layer-4-Hypothese)')

---
## 4. Zustandsbewertung & Übergabe an Layer 7

In [ ]:
# ============================================================
# LAYER-6-SCORE
# ============================================================

def norm(v, lo, hi):
    if v is None: return None
    return round(max(0.0, min(1.0, (v - lo) / (hi - lo))), 4)

# 1) Anregungs-Stärke (aus Amplituden-Modulator)
excitation = norm(amp_mod_total, 0.5, 1.8)

# 2) Frequenz-Anomalie (Gesamtverschiebung der SR-1)
freq_anomaly = norm(abs(expected[1]['freq_shift_mHz']), 0, 200)

# 3) Q-Faktor-Abweichung
q_anomaly = norm(abs(expected[1]['Q_factor_change']), 0, 30)

# 4) Chimney-Aktivität
chimney_score = norm(chimney_factor, 0.85, 1.15)

# 5) Nicht-geometrische Dominanz (Diagnose-Feature)
non_geom_dom = norm(ratio, 0, 10)

COMPONENTS = {
    'Anregungs-Staerke (Amplitude)':  {'score': excitation,    'source': 'derived_from_L3_L5', 'dynamic': True},
    'Frequenz-Anomalie (SR-1)':       {'score': freq_anomaly,  'source': 'derived_from_L4',   'dynamic': True},
    'Q-Faktor-Abweichung':            {'score': q_anomaly,     'source': 'derived_from_L4',   'dynamic': True},
    'Chimney-Aktivitaet':             {'score': chimney_score, 'source': 'time_of_day',       'dynamic': True},
    'Nicht-geom. Dominanz':           {'score': non_geom_dom,  'source': 'diagnostic',        'dynamic': True},
}

available    = {k: v['score'] for k, v in COMPONENTS.items() if v['score'] is not None}
unavailable  = [k for k, v in COMPONENTS.items() if v['score'] is None]
layer6_score = round(sum(available.values()) / len(available), 4) if available else None
confidence   = round(len(available) / len(COMPONENTS), 2)
level = ('unbekannt' if layer6_score is None
         else 'ruhig'   if layer6_score < 0.3
         else 'moderat' if layer6_score < 0.6
         else 'aktiv')
dominant_l6 = max(available, key=available.get) if available else 'none'

W = 70
print('=' * W)
print('LAYER 6 – RESONANZFELD / SCHUMANN-RESONANZ – ZUSTANDSBEWERTUNG')
print('=' * W)
for name, comp in COMPONENTS.items():
    s = comp['score']
    if s is not None:
        bar = '█' * int(s*20) + '░' * (20-int(s*20))
        print(f'  {name:<32} {bar}  {s:.3f}  [{comp["source"]}]')
    else:
        print(f'  {name:<32} {"─"*20}  n/a   [missing]')
print('-' * W)
print(f'  Score:           {layer6_score:.3f}  ({len(available)}/{len(COMPONENTS)} Komponenten)')
print(f'  Confidence:      {confidence:.0%}')
print(f'  Level:           {level.upper()}')
print(f'  Dominant:        {dominant_l6}')
print(f'  SR-1 erwartet:   {expected[1]["freq_Hz"]:.4f} Hz  (Δ {expected[1]["freq_shift_mHz"]:+.1f} mHz vs 7.83 Hz)')
print(f'  Amplitude:       Faktor {amp_mod_total:.3f}  (vs Referenz)')
print(f'  Q-Faktor SR-1:   {expected[1]["Q_factor"]:.2f}  ({expected[1]["Q_factor_change"]:+.1f}%)')
print('=' * W)

# Radar
cats   = list(available.keys())
vals_r = list(available.values())
if len(cats) >= 3:
    fig = go.Figure()
    fig.add_trace(go.Scatterpolar(
        r=vals_r + [vals_r[0]], theta=cats + [cats[0]],
        fill='toself', fillcolor='rgba(83,74,183,0.20)',
        line=dict(color='#534AB7', width=2.5), name='Layer 6'
    ))
    fig.add_trace(go.Scatterpolar(
        r=[0.5]*(len(cats)+1), theta=cats+[cats[0]],
        line=dict(color='#E24B4A', dash='dot', width=1),
        mode='lines', name='Aktivitätsschwelle'
    ))
    fig.update_layout(
        title=dict(
            text=f'Layer 6 – Resonanzfeld | Score: {layer6_score:.3f} | {level.upper()} | Confidence: {confidence:.0%}',
            font=dict(size=12)
        ),
        polar=dict(radialaxis=dict(visible=True, range=[0,1])),
        height=450, showlegend=True,
        margin=dict(l=80, r=80, t=70, b=40)
    )
    fig.show()

In [ ]:
# ============================================================
# EXPORT – layer6_state.json
# ============================================================

_amp_str  = ('erhoehte Anregung' if amp_mod_total > 1.2
             else 'gedaempfte Anregung' if amp_mod_total < 0.85
             else 'normale Anregung')
_freq_str = ('Frequenzverschiebung deutlich' if abs(expected[1]['freq_shift_mHz']) > 100
             else 'Frequenz nahe Referenz')

state_summary = (
    f'Layer-6-Zustand: {level}. '
    f'SR-1 erwartet {expected[1]["freq_Hz"]:.3f} Hz '
    f'(Δ {expected[1]["freq_shift_mHz"]:+.1f} mHz vs Referenz 7.83 Hz). '
    f'Amplitude-Faktor {amp_mod_total:.2f} ({_amp_str}). '
    f'Q-Faktor SR-1 {expected[1]["Q_factor"]:.2f} ({expected[1]["Q_factor_change"]:+.1f}%). '
    f'Aktive Chimneys: {[c[0] for c in active_chimneys] if active_chimneys else "–"}. '
    f'Nicht-geometrische Dominanz: {ratio:.1f}x. '
    f'Datenvollstaendigkeit: {confidence:.0%}.'
)

layer6_state = {
    'timestamp': datetime.datetime.utcnow().isoformat() + 'Z',
    'layer': 6,
    'name':  'Resonanzfeld / Schumann-Resonanz',

    # Wichtig: alle Werte sind modellierte Erwartung, keine reale Messung
    'measurement_status':      'model_expected_not_observed',
    'observed_data_available': False,
    'observed': {
        'SR_1_freq_Hz':    None,
        'SR_1_amplitude_pT': None,
        'Q_factor':        None,
        'note': 'Hier reale Messdaten eintragen wenn verfügbar (z.B. AURA, NASA/MEPAS)'
    },

    'score':      layer6_score,
    'level':      level,
    'confidence': confidence,
    'score_basis': f'{len(available)}/{len(COMPONENTS)} dynamische Komponenten',
    'dominant_component': dominant_l6,
    'missing_components': unavailable,

    'components': {
        k: {'score': round(v['score'], 4) if v['score'] is not None else None,
            'source': v['source'], 'dynamic': v['dynamic']}
        for k, v in COMPONENTS.items()
    },

    # Empirische Referenz (immer verfügbar)
    'empirical_reference': {
        f'SR_{n}': {**v, 'unit': {'freq_Hz': 'Hz', 'amplitude_pT': 'pT', 'Q_factor': '–'}}
        for n, v in SR_REF.items()
    },

    # Modulierte Erwartung pro Modus
    'expected_modulated': {
        f'SR_{n}': v for n, v in expected.items()
    },

    # Modulator-Aufschlüsselung (für Layer-7-Engine)
    'modulators': {
        'frequency_shifts_mHz': {
            'geometric_from_L4': geom_delta_mhz,
            'day_night':         day_night_shift_mhz,
            'kp_effect':         kp_shift_mhz,
            'xray_effect':       xray_shift_mhz,
        },
        'amplitude_factor': {
            'total':           round(amp_mod_total, 4),
            'gec_generator':   round(l5_generator, 3),
            'thunder_l3':      round(1.0 + (l3_thunder_score or 0.15) * 0.5, 3),
            'chimney_factor':  round(chimney_factor, 3),
        },
        'q_factor_modulation': {
            'total':         round(q_mod, 4),
            'ionization':    round(1.0 - (l4_ioniz_score or 0.3) * 0.15, 4),
            'xray_penalty':  0.7 if l4_xray_class in ['M','X'] else 1.0,
        },
    },

    # Tagesmuster (Chimneys)
    'diurnal_pattern': {
        'utc_hour': utc_hour,
        'active_chimneys': [{'name': n, 'activity': round(a, 3)} for n, a in active_chimneys],
        'global_activity': round(chimney_activity, 3),
    },

    # Delta-Diagnose – Kernoutput für Layer 7
    # Welcher nicht-geometrische Faktor ist aktuell wirklich führend?
    'dominant_physical_driver': (
        'kp_geomagnetic'        if abs(kp_shift_mhz) > abs(day_night_shift_mhz) and abs(kp_shift_mhz) > 30
        else 'solar_xray_flare' if abs(xray_shift_mhz) > 20
        else 'diurnal_chimney_distribution'
    ),
    'delta_analysis': {
        'geometric_total_SR1_mHz':      round(geom_total_sr1, 2),
        'non_geometric_total_SR1_mHz':  round(non_geom_total_sr1, 2),
        'ratio_non_geom_to_geom':       round(ratio, 2),
        'interpretation': (
            'non_geometric_dominant' if ratio > 3
            else 'mixed' if ratio > 1
            else 'geometric_dominant'
        ),
        'non_geometric_dominance_level': (
            'strong_confirmed'  if ratio > 5
            else 'weak_confirmed' if ratio > 3
            else 'marginal'       if ratio > 2
            else 'absent'
        ),
        'non_geometric_dominance_margin': round(ratio - 3.0, 3),
        'note': (
            'Nicht-geometrische Faktoren (Tag/Nacht, Kp, X-Ray) verschieben '
            'die Resonanz deutlich staerker als die reine Cavity-Hoehe (Layer 4).'
        ) if ratio > 3 else 'Geometrische und nicht-geometrische Effekte ausgeglichen.'
    },

    'flags': {
        'amplitude_elevated':    bool(amp_mod_total > 1.2),
        'amplitude_suppressed':  bool(amp_mod_total < 0.85),
        'frequency_anomaly':     bool(abs(expected[1]['freq_shift_mHz']) > 100),
        'q_factor_degraded':     bool(expected[1]['Q_factor_change'] < -10),
        'chimney_active':        bool(len(active_chimneys) > 0),
        'non_geometric_dominant': bool(ratio > 3),
    },

    'thresholds': {
        'amp_elevated':       1.2,
        'amp_suppressed':     0.85,
        'freq_anomaly_mHz':   100.0,
        'q_degraded_pct':     -10.0,
        'non_geom_ratio':     3.0,
    },
    'level_thresholds': {
        'ruhig':   [0.0,  0.30],
        'moderat': [0.30, 0.60],
        'aktiv':   [0.60, 0.80],
        'stark':   [0.80, 1.0],
        'note': 'Score 0.30 ist untere Grenze moderat; Werte nahe Schwelle als ruhig_bis_moderat interpretieren'
    },

    'downstream_expectation': {
        'layer7_engine': (
            f'L6-Score {layer6_score:.3f} ({level}) – '
            f'SR-1 {expected[1]["freq_Hz"]:.2f} Hz | '
            f'Amp {amp_mod_total:.2f}x | '
            f'non_geom_dom={ratio:.1f}x'
        ),
        'layer8_research': (
            'Hypothese: Wenn reale Messungen nicht_geometrische_dominanz bestaetigen, '
            'sind aktuell vor allem Tag/Nacht-Struktur, Leitfaehigkeit/Daempfung '
            'und Quellenverteilung (Chimneys) fuehrend. '
            'Kp und Solarflares werden nur bei aktiver Stoerung relevant.'
        ),
    },

    'layer_context': {
        'L0': {'score': L0['score'], 'level': L0['level']} if L0 else None,
        'L1': {'score': L1['score'], 'level': L1['level']} if L1 else None,
        'L2': {'score': L2['score'], 'level': L2['level']} if L2 else None,
        'L3': {'score': L3['score'], 'level': L3['level'],
               'thunder_score': l3_thunder_score,
               'CAPE_mean': l3_cape,
               'n_thunder': l3_n_thunder} if L3 else None,
        'L4': {'score': L4['score'], 'level': L4['level'],
               'cavity_height_km': l4_cavity_h,
               'cavity_delta_km': l4_cavity_delta,
               'day_night': l4_day_night,
               'cavity_delta_insight': L4.get('resonance_system',{}).get('cavity_delta_insight')} if L4 else None,
        'L5': {'score': L5['score'], 'level': L5['level'],
               'V_iono_kV': l5_v_iono,
               'delta_V_pct': l5_delta_v_pct,
               'generator_strength': l5_generator} if L5 else None,
    },

    'state_summary': state_summary,
}

# numpy-Bereinigung
def _to_python(obj):
    if isinstance(obj, dict):  return {k: _to_python(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [_to_python(v) for v in obj]
    if isinstance(obj, np.bool_):    return bool(obj)
    if isinstance(obj, np.integer):  return int(obj)
    if isinstance(obj, np.floating): return None if np.isnan(obj) else float(obj)
    return obj
layer6_state = _to_python(layer6_state)

with open(layer_state(6), 'w', encoding='utf-8') as f:
    json.dump(layer6_state, f, indent=2, ensure_ascii=False)

print(f'gespeichert: {layer_state(6)}')
print(json.dumps(layer6_state, indent=2, ensure_ascii=False))


---
## Zusammenfassung Layer 6

| Aspekt | Inhalt |
|--------|--------|
| **Rolle** | Beobachtbares EM-Muster des Gesamtsystems – Schumann-Resonanz |
| **Empirische Basis** | SR-1 7.83 Hz, SR-2 14.3 Hz, SR-3 20.8 Hz, SR-4 27.3 Hz, SR-5 33.8 Hz |
| **Frequenz-Modulatoren** | geometrisch (L4) + Tag/Nacht + Kp + X-Ray |
| **Amplituden-Modulatoren** | Generator (L5), Gewitter (L3), Chimney (UTC) |
| **Q-Faktor-Modulator** | Ionosphärische Leitfähigkeit (L4) |
| **Tagesmuster** | 3 Chimneys: Asien (8h), Afrika (14h), Amerika (20h UTC) |
| **Delta-Diagnose** | `non_geometric_dominance = real_Δ / geometric_Δ` |
| **→ Layer 7** | `expected_modulated`, `modulators`, `delta_analysis` als Engine-Input |
| **→ Layer 8** | Diagnose-Hypothese als Forschungs-Trigger |
| **Ausgabe** | `layer6_state.json` mit vollständigem Resonanzmuster |

> **Nächster Schritt:** `atmosphere_analysis_layer7.ipynb` – Earth Field State Engine